# Flux post-processing chain: both gas analysers merged (2004-2026.09)

**notebook version**: `1` (29 Oct 2026)

## ℹ️ About this notebook

Joins the merged flux results of the two gas analysers into one record, 2004 to September 2026:

| Period | Source | Analyser |
|---|---|---|
| 1 January 2004 - 31 December 2015 | `IRGA75/99_FPPC_IRGA75_MERGED_2004-2017` | open-path LI-7500, CO2 flux corrected with opco |
| 1 January 2016 - 30 September 2026 | `IRGA72/99_FPPC_IRGA72_MERGED_2016-2026.09` | enclosed-path LI-7200 |

Both files carry the same column names, so every column holds one analyser per period. The column
`FLAG_FLUX_SOURCE` records it for each half-hour: 75 for the LI-7500, 72 for the LI-7200. In the
overlap 2016-2017 the LI-7200 is used. The LI-7500 stand-in run of 17 January to 20 March 2019 is
not used; that period comes from the LI-7200 chain, gap-filled.

Values are copied unchanged. Columns that only one analyser has (the manual removal flags of the
LI-7200) are empty in the other period. Known limitations of the LI-7500 years are on the site
documentation (Issues).

## ⚙️ Settings

In [ ]:
DATADIR = (r"F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data"
           r"\workflow")
STAGEDIR = rf"{DATADIR}\30_FLUX_POST-PROCESSING_CHAIN"
FILES = {
    'IRGA75': rf"{STAGEDIR}\IRGA75\99_FPPC_IRGA75_MERGED_2004-2017.parquet",
    'IRGA72': rf"{STAGEDIR}\IRGA72\99_FPPC_IRGA72_MERGED_2016-2026.09.parquet",
}
# First half-hour of each analyser in the merged record (TIMESTAMP_MIDDLE).
PERIODS = {'IRGA75': ('2004-01-01 00:15', '2015-12-31 23:45'),
           'IRGA72': ('2016-01-01 00:15', '2026-09-30 23:45')}
SOURCE_CODES = {'IRGA75': 75, 'IRGA72': 72}
SOURCE_COL = 'FLAG_FLUX_SOURCE'

OUTNAME = "99_FPPC_MERGED_2004-2026.09"  # same name as this notebook
OUTPATH = STAGEDIR

UMOL_TO_GC = 12.011 * 1e-6 * 1800  # NEE, umol m-2 s-1 -> gC m-2 per half-hour
WM2_TO_MJ = 1800 * 1e-6  # LE and H, W m-2 -> MJ m-2 per half-hour

In [ ]:
from datetime import datetime
import importlib.metadata
import textwrap

import diive as dv
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from diive.core.plotting.styles import chainplots as cp


def figcap(fig, text, width=150):
    """Place a caption under a figure drawn with a layout engine. Call after the last axes."""
    width = min(width, max(40, int(14.5 * fig.get_size_inches()[0])))
    fig.supxlabel(textwrap.fill(f"Figure - {text}", width), fontsize=9, ha='left', x=0.005)


NOTEBOOK_START = datetime.now()
print(f"Last run: {NOTEBOOK_START.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"diive version: v{importlib.metadata.version('diive')}")

## 📂 Load data

In [ ]:
frames = {k: dv.load_parquet(filepath=v) for k, v in FILES.items()}
for name, frame in frames.items():
    _start, _end = (pd.Timestamp(t) for t in PERIODS[name])
    assert frame.index[0] <= _start and frame.index[-1] >= _end, f"{name} does not cover {PERIODS[name]}"
    assert not frame.index.duplicated().any(), f"{name} has duplicate timestamps"
    print(f"{name}: {frame.shape[0]:,} records x {frame.shape[1]} columns, {frame.index[0]} -> {frame.index[-1]}")
_a, _b = (set(frames[k].columns) for k in FILES)
print(f"columns only in IRGA75: {sorted(_a - _b)}")
print(f"columns only in IRGA72: {sorted(_b - _a)}")

## 🔗 Merge

In [ ]:
index = pd.date_range(PERIODS['IRGA75'][0], PERIODS['IRGA72'][1], freq='30min', name='TIMESTAMP_MIDDLE')
columns = list(frames['IRGA72'].columns) + [c for c in frames['IRGA75'].columns if c not in frames['IRGA72'].columns]
parts, source = [], pd.Series(np.nan, index=index, name=SOURCE_COL)
for name, (start, end) in PERIODS.items():
    _part = frames[name].loc[start:end].reindex(columns=columns)
    parts.append(_part)
    source.loc[start:end] = SOURCE_CODES[name]
merged = pd.concat(parts).reindex(index)
merged[SOURCE_COL] = source.astype('int64')
print(f"Merged: {merged.shape[0]:,} records x {merged.shape[1]} columns, {merged.index[0]} -> {merged.index[-1]}")

## 🔍 Checks

In [ ]:
assert merged.index.equals(index) and not merged.index.duplicated().any()
assert merged[SOURCE_COL].notna().all()
# Every value comes unchanged from the analyser of its period.
for name, (start, end) in PERIODS.items():
    _src = frames[name].loc[start:end]
    for c in _src.columns:
        assert merged.loc[start:end, c].equals(_src[c].reindex(merged.loc[start:end].index)), f"{c} changed ({name})"
# The main gap-filled series are complete over the whole record.
MAIN = ['NEE_L3.1_L3.3_CUT_50_QCF_gfXG', 'LE_L3.1_L3.3_CUT_NONE_QCF_gfXG', 'ET_L3.1_L3.3_CUT_NONE_QCF_gfXG',
        'H_L3.1_L3.3_CUT_NONE_QCF_gfXG']
for c in MAIN:
    assert merged[c].notna().all(), f"{c} has gaps"
print("Index, sources and values verified; main gap-filled series complete:", ", ".join(MAIN))

### 📋 Table: half-hours per analyser and year

In [ ]:
display(merged[SOURCE_COL].groupby(merged.index.year).value_counts().unstack(fill_value=0)
        .rename(columns={v: k for k, v in SOURCE_CODES.items()}))

### 📋 Table: annual sums of the gap-filled fluxes

`NEE` uses the CUT_50 scenario; `LE`, `ET` and `H` are not USTAR-filtered (`CUT_NONE`). 2026 ends on 30 September.

In [ ]:
_vars = {'NEE (gC m-2)': ('NEE_L3.1_L3.3_CUT_50_QCF_gf{m}', UMOL_TO_GC),
         'LE (MJ m-2)': ('LE_L3.1_L3.3_CUT_NONE_QCF_gf{m}', WM2_TO_MJ),
         'ET (mm)': ('ET_L3.1_L3.3_CUT_NONE_QCF_gf{m}', 1),
         'H (MJ m-2)': ('H_L3.1_L3.3_CUT_NONE_QCF_gf{m}', WM2_TO_MJ)}
_year = merged.index.year
ANNUAL = pd.DataFrame({(_v, _mn): (merged[_p.format(m=_m)] * _g).groupby(_year).sum()
                       for _v, (_p, _g) in _vars.items() for _mn, _m in (('XGBoost', 'XG'), ('MDS', 'MDS'))})
ANNUAL.insert(0, 'analyser', merged[SOURCE_COL].groupby(_year).first().map({v: k for k, v in SOURCE_CODES.items()}))
ANNUAL.index.name = 'year'
display(ANNUAL.round(0))

### 📊 Plot: cumulative NEE over the merged record

In [ ]:
_cols = {s: f'NEE_L3.1_L3.3_{s}_QCF_gfXG' for s in ('CUT_16', 'CUT_50', 'CUT_84')}
_cum = {s: (merged[c] * UMOL_TO_GC).cumsum() for s, c in _cols.items()}
_switch = pd.Timestamp(PERIODS['IRGA72'][0])
fig, axs = plt.subplots(2, 1, figsize=(16, 9.5), layout='constrained', height_ratios=[1.6, 1])
ax = axs[0]
ax.fill_between(_cum['CUT_50'].index, np.minimum(_cum['CUT_16'], _cum['CUT_84']),
                np.maximum(_cum['CUT_16'], _cum['CUT_84']), color=cp.KEPT, alpha=0.5, linewidth=0,
                label='CUT_16 to CUT_84')
ax.plot(_cum['CUT_50'].index, _cum['CUT_50'].to_numpy(), color=cp.INK, linewidth=1.8,
        label=f"NEE, CUT_50, XGBoost: {_cum['CUT_50'].iloc[-1]:,.0f}")
ax.axvline(_switch, color=cp.MUTED, linewidth=1, linestyle=':')
ax.text(0.01, 0.97, 'IRGA75, LI-7500 (opco)', transform=ax.transAxes, va='top', fontsize=9, color=cp.MUTED)
ax.text(_switch, 0.97, '  IRGA72, LI-7200', transform=ax.get_xaxis_transform(), va='top', fontsize=9, color=cp.MUTED)
ax.axhline(0, color=cp.INK, linewidth=0.8, alpha=0.5)
ax.set_ylabel('cumulative NEE since 1 Jan 2004 (gC m-2)')
cp.style_axes(ax)
ax.legend(frameon=False, fontsize=9, loc='lower left')
cp.panel_label(ax, 'cumulative NEE')
ax = axs[1]
_ann = ANNUAL[('NEE (gC m-2)', 'XGBoost')]
_x = _ann.index.to_numpy()
ax.bar(_x, _ann.to_numpy(), width=0.7, color=np.where(_x < _switch.year, cp.REJECTED, cp.CLEANED))
ax.axhline(0, color=cp.INK, linewidth=0.8, alpha=0.5)
ax.set_xticks(_x, [str(v) for v in _x], rotation=90)
ax.set_ylabel('annual NEE (gC m-2)')
cp.style_axes(ax)
cp.panel_label(ax, 'annual NEE, CUT_50, XGBoost (red: IRGA75, blue: IRGA72; 2026 to 30 September)')
cp.figure_title(fig, 'NEE 2004 - Sep 2026 from both gas analysers', 'Negative = uptake.')
figcap(fig, "the cumulative and the annual NEE of the merged record: the LI-7500 until 2015, the LI-7200 from "
            "2016. The gap-filled annual NEE of the LI-7500 years may show too much uptake, because that "
            "analyser misses many wet half-hours (see the site documentation, Issues).")
plt.show()

### 📊 Plot: daily time series of the main fluxes, 2004 - Sep 2026

In [ ]:
# Daily values of the gap-filled series (XGBoost), with a 30-day running mean. The shading marks
# the LI-7500 period.
SERIES = {'NEE': ('NEE_L3.1_L3.3_CUT_50_QCF_gfXG', 'mean', 'daily mean (µmol m-2 s-1)'),
          'LE': ('LE_L3.1_L3.3_CUT_NONE_QCF_gfXG', 'mean', 'daily mean (W m-2)'),
          'ET': ('ET_L3.1_L3.3_CUT_NONE_QCF_gfXG', 'sum', 'daily sum (mm)'),
          'H': ('H_L3.1_L3.3_CUT_NONE_QCF_gfXG', 'mean', 'daily mean (W m-2)')}
fig, axs = plt.subplots(len(SERIES), 1, figsize=(16, 3.2 * len(SERIES) + 0.8), sharex=True, layout='constrained')
for ax, (name, (col, how, unit)) in zip(axs, SERIES.items()):
    _d = getattr(merged[col].resample('D'), how)()
    ax.axvspan(pd.Timestamp(PERIODS['IRGA75'][0]), _switch, color=cp.REJECTED, alpha=0.06, linewidth=0)
    ax.plot(_d.index, _d.to_numpy(), color=cp.KEPT, linewidth=0.5, label='daily')
    ax.plot(_d.index, _d.rolling(30, center=True, min_periods=15).mean().to_numpy(), color=cp.INK,
            linewidth=1.2, label='30-day running mean')
    ax.axvline(_switch, color=cp.MUTED, linewidth=1, linestyle=':')
    ax.axhline(0, color=cp.INK, linewidth=0.8, alpha=0.5)
    ax.set_ylabel(unit)
    cp.style_axes(ax)
    cp.panel_label(ax, f"{name}, {col}")
axs[0].legend(frameon=False, fontsize=9, loc='lower left', ncol=2)
cp.figure_title(fig, 'Main fluxes 2004 - Sep 2026, gap-filled with XGBoost',
                'Shaded: LI-7500 (IRGA75) until 2015; LI-7200 (IRGA72) from 2016. NEE: CUT_50.')
figcap(fig, "daily values of the gap-filled NEE, LE, ET and H of the merged record, with their 30-day running "
            "mean. The LI-7500 years are shaded.")
plt.show()

### 📊 Plot: heatmaps of the main fluxes, date against time of day

In [ ]:
# One panel per flux, the gap-filled half-hourly values (XGBoost); colour scales at the 1st and 99th
# percentiles (NEE symmetric around zero).
_heat = {'NEE (µmol m-2 s-1)': SERIES['NEE'][0], 'LE (W m-2)': SERIES['LE'][0], 'H (W m-2)': SERIES['H'][0]}
fig, axs = plt.subplots(1, len(_heat), figsize=(17.6, 16), sharey=True, layout='constrained')
for _j, (ax, (title, col)) in enumerate(zip(axs, _heat.items())):
    _s = merged[col]
    _lo, _hi = (float(v) for v in _s.quantile([0.01, 0.99]))
    _lim = max(abs(_lo), abs(_hi)) if title.startswith('NEE') else None
    dv.plotting.HeatmapDateTime(series=_s, verbose=False).plot(
        ax=ax, fig=fig, vmin=-_lim if _lim else _lo, vmax=_lim if _lim else _hi, cmap=cp.HEATMAP_CMAP,
        cb_extend='both', cb_digits_after_comma='auto', cb_labelsize=8, zlabel=title,
        color_bad=cp.NAN_COLOR, as_image=True, format_style=cp.heatmap_style(title))
    if _j:
        plt.setp(ax.get_yticklabels(), visible=False)
        ax.yaxis.get_label().set_visible(False)
cp.figure_title(fig, 'Main fluxes 2004 - Sep 2026, half-hourly, gap-filled',
                'Date (bottom to top) against time of day. LI-7500 until 2015, LI-7200 from 2016.')
plt.show()

## 💾 Export

The merged file as parquet, and next to it an `_INFO.md` file that gives the source of each period and lists every column.

In [ ]:
filepath = dv.save_parquet(data=merged, filename=OUTNAME, outpath=OUTPATH)
_lines = [f"# {OUTNAME}", "",
          f"{merged.index[0]} to {merged.index[-1]} ({merged.index.name}, UTC+01:00), "
          f"{merged.shape[0]:,} records x {merged.shape[1]} columns.",
          f"Written {datetime.now():%Y-%m-%d %H:%M} by `{OUTNAME}.ipynb`.", "",
          "| period | analyser | source file | `FLAG_FLUX_SOURCE` |", "|---|---|---|---|"]
_lines += [f"| {s} - {e} | {k} | `{FILES[k]}` | {SOURCE_CODES[k]} |" for k, (s, e) in PERIODS.items()]
_lines += ["", f"## Columns ({merged.shape[1]})", ""] + [f"- `{c}`" for c in merged.columns]
with open(rf"{OUTPATH}\{OUTNAME}_INFO.md", "w", encoding="utf-8") as _f:
    _f.write("\n".join(_lines) + "\n")
print(f"Saved to: {filepath}")
print(f"Wrote {OUTPATH}\\{OUTNAME}_INFO.md")

## ✅ Read the written file back

In [ ]:
_written = dv.load_parquet(filepath=rf"{OUTPATH}\{OUTNAME}.parquet")
assert _written.index.equals(merged.index), "index changed on write"
assert list(_written.columns) == list(merged.columns), "columns changed on write"
for _c in MAIN + [SOURCE_COL]:
    assert _written[_c].equals(merged[_c]), f"{_c} changed on write"
print(f"Written file verified: {_written.shape[0]:,} records x {_written.shape[1]} columns")

***

# End of notebook.

In [ ]:
NOTEBOOK_END = datetime.now()
print(f"Runtime: {NOTEBOOK_END - NOTEBOOK_START}")